In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

!pwd
import os
os.chdir('/content/drive/My Drive/Colab Notebooks')
!pwd
# Get the current working directory
current_directory = os.getcwd()

# Specify the file name
file_name = "mastershifted.txt"

# Construct the full path to the file
file_path = os.path.join(current_directory, file_name)

# Open the file
try:
    with open(file_path, "r") as f:
        # Do something with the file (e.g., print its contents)
        print("Loaded successfully")
except FileNotFoundError:
    print(f"The file '{file_name}' was not found in the current directory.")
except Exception as e:
    print(f"An error occurred: {e}")

/content
/content/drive/My Drive/Colab Notebooks
Loaded successfully


In [ ]:
import numpy as np
import tensorflow as tf
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Reshape, MaxPooling2D, Dense, Dropout, Activation, Bidirectional, Flatten, BatchNormalization
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

In [ ]:
colnames=( 'Bx', 'By', 'Bz', 'Vx', 'Density', 'Labels')
df = pd.read_csv('mastershifted.txt', sep='\s+', names=colnames)
dfc=df.drop(['Labels'], axis=1)

n_timesteps = 120

Xcols=[x for x in dfc.columns if x!= 'Labels']
n_features=len(Xcols)
model= Sequential()
scaler = MinMaxScaler()
X = df[Xcols].values
y=df['Labels']
targtrain=np.empty((53404,))
targtrain[::2]=0 #BE VERY CAREFUL WITH THE 1S AND OS in [::2] and [1::2]
targtrain[1::2]=1 #BE VERY CAREFUL WITH THE 1S AND OS in [::2] and [1::2]

targtest=np.empty((11444,))
targtest[::2]=0 #BE VERY CAREFUL WITH THE 1S AND OS in [::2] and [1::2]
targtest[1::2]=1 #BE VERY CAREFUL WITH THE 1S AND OS in [::2] and [1::2]

targvalid=np.empty((11444,))
targvalid[::2]=0 #BE VERY CAREFUL WITH THE 1S AND OS in [::2] and [1::2]
targvalid[1::2]=1 #BE VERY CAREFUL WITH THE 1S AND OS in [::2] and [1::2]

X_train = X[0:6408480]
y_train = targtrain
X_test =  X[6408480:7781760]
y_test =  targtest
X_val =   X[7781760:9155040]
y_val =   targvalid


In [ ]:
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_val=scaler.transform(X_val)

n_samples = int(X_train.shape[0]/120)
X_tr = np.resize(X_train, (n_samples, n_timesteps, n_features))
X_te = np.resize(X_test, (11444, 120, 5))
X_va = np.resize(X_val, (11444, 120, 5))

In [ ]:
model = Sequential()
model.add(Bidirectional(LSTM(16, return_sequences=True)))
model.add(Bidirectional(LSTM(32, return_sequences=False)))
model.add(BatchNormalization())
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer=Adam(learning_rate=0.000001), loss='binary_crossentropy', metrics=['accuracy'])
history=model.fit(X_tr, y_train, epochs=70, batch_size=128, validation_data=(X_va, y_val))

In [ ]:
# Crucially, we need the output before the final Dense layer,
# since we will use this data for binary classification with XGB

feature_extractor = Sequential(model.layers[:-3]) #-2=BiLSTM32 works, -3=BiLSTM16 works

cnn_lstm_train_output = feature_extractor.predict(X_tr)
predictions_train = cnn_lstm_train_output.reshape(cnn_lstm_train_output.shape[0], -1)

cnn_lstm_test_output = feature_extractor.predict(X_te)  # Shape: (50000, 120
predictions_test = cnn_lstm_test_output.reshape(cnn_lstm_test_output.shape[0], -1)

cnn_lstm_val_output = feature_extractor.predict(X_va)
predictions_val = cnn_lstm_val_output.reshape(cnn_lstm_val_output.shape[0], -1)

# Now we will feed the model's output into XGBoost for binary classification based on:
# the temporal features from LSTM

In [ ]:
import xgboost as xgb

# Convert the flattened data into DMatrix (XGBoost's internal data format)
dtrain = xgb.DMatrix(predictions_train, label=y_train)
dtest = xgb.DMatrix(predictions_test, label=y_test)
dvalid = xgb.DMatrix(predictions_val, label=y_val)

In [ ]:
##this works for the BiLSTM 32 layer and 16 layer

params = {
    'objective': 'binary:logistic',  # Binary classification task
    'eval_metric': ['logloss'],        # Log-loss metric for binary classification
    #'gamma':5,
    'max_depth': 5,
    'learning_rate': 0.005,
    'lambda': 100,
    'alpha': 100,
}

evals = [(dtrain, 'train'), (dtest, 'test'), (dvalid, 'validation')]
evals_result = {}

# Train the XGBoost model #
xgbmodel = xgb.train(params, dtrain, num_boost_round=600,evals=evals,early_stopping_rounds=10, evals_result=evals_result,  verbose_eval=True)

epochs = range(1, len(evals_result['train']['logloss']) + 1)
train_logloss = evals_result['train']['logloss']
test_logloss = evals_result['test']['logloss']
val_logloss = evals_result['validation']['logloss']

plt.plot(epochs, train_logloss, label='Train Logloss')
#plt.plot(epochs, test_logloss, label='Test Logloss')
plt.plot(epochs, val_logloss, label='Validation Logloss')
plt.xlabel('Boosting Round')
plt.ylabel('Logloss')
plt.title('Smoothed LogLoss vs. Epoch')
plt.legend()
plt.show()

In [ ]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.linear_model import LogisticRegression

# Get the raw prediction scores (logits) from XGBoost
#raw_preds = xgbmodel.predict(dvalid, output_margin=True)  # Get the raw logits, not probabilities
raw_pred_train = xgbmodel.predict(dtrain, output_margin=True)
raw_pred_test = xgbmodel.predict(dtest, output_margin=True)
raw_pred_val = xgbmodel.predict(dvalid, output_margin=True)


# Apply Platt Scaling via sklearn's CalibratedClassifierCV
# First, you need to fit a logistic regression model to these raw predictions
log_reg = LogisticRegression()
log_reg.fit(raw_pred_train.reshape(-1, 1), y_train)
log_reg.fit(raw_pred_test.reshape(-1, 1), y_test)
log_reg.fit(raw_pred_val.reshape(-1, 1), y_val)

# Then, we calibrate the predictions
calibrated_pred_train = log_reg.predict_proba(raw_pred_train.reshape(-1, 1))[:, 1]
calibrated_pred_test = log_reg.predict_proba(raw_pred_test.reshape(-1, 1))[:, 1]
calibrated_pred_val = log_reg.predict_proba(raw_pred_val.reshape(-1, 1))[:, 1]  # Get probabilities for the positive class



In [ ]:
from sklearn.metrics import log_loss
log_loss_before = log_loss(y_test, raw_pred_test)
print(f"Log Loss before Platt Calibration: {log_loss_before}")

log_loss_after = log_loss(y_test, calibrated_pred_test)
print(f"Log Loss after Platt Calibration: {log_loss_after}")
